# Split del dataset de perfiles completos

En este notebook se genera una metadata persistente con una partición estratificada.

El objetivo es que otros notebooks puedan reutilizar exactamente la misma división `train` / `val` / `test` sin recalcularla.

El resultado se guarda como `metadata.csv` con una nueva columna `split`.

In [1]:
# Importaciones
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from torchvision import datasets

from vasijas.config import load_config
from vasijas.entrenamiento_cnn import ConfiguracionSplitDataset
from vasijas.metadata import cargar_metadata, guardar_metadata
from vasijas.paths import ProjectPaths

In [9]:
# Configuracion general
CONFIG = load_config()
PATHS = ProjectPaths.from_root(CONFIG.project_root)

DATA_ORIGEN = PATHS.DATA_CUADRADOS
DATA_DESTINO = PATHS.DATA_DIVIDIDO

METADATA_ORIGEN = cargar_metadata(DATA_ORIGEN)

print(f"Dataset de imágenes: {DATA_ORIGEN}")
print("Muestra metadata origen:")
print(METADATA_ORIGEN.head())

Dataset de imágenes: C:\Users\rafae\Dev\vasijas\data\05.cuadrados
Muestra metadata origen:
  Identificador Tiene asa Tendencia Formal  Borde Cuello  Cuerpo  Base  \
0        al_03a        No               II      1      B       2     2   
1        al_03b        No               II      1      B       2     2   
2        al_03c        No               II      1      B       2     2   
3        al_03d        No               II      1      B       2     2   
4        al_03e        No               II      1      B       2     2   

   Clase Ana  Diametro Borde  Diametro Base  Altura Total Cronologia  \
0       11.0             182             54            53      ibero   
1       11.0             178             57            46      ibero   
2       11.0             185             56            49      ibero   
3       11.0             160             41            45      ibero   
4       11.0             164             54            44      ibero   

     Ubicacion  
0  Almedinilla

In [ ]:
# Parámetros del split
SEED = 42
config_split = ConfiguracionSplitDataset(semilla=SEED)

# Usar None para procesar todas las clases, o una lista con los nombres de carpetas.
CLASES_A_PROCESAR = None
# CLASES_A_PROCESAR = ["clase_1", "clase_2"]

print(
    "Split configurado -> "
    f"train: {1 - config_split.proporcion_temporal:.0%}, "
    f"val: {config_split.proporcion_temporal * (1 - config_split.proporcion_test_desde_temporal):.0%}, "
    f"test: {config_split.proporcion_temporal * config_split.proporcion_test_desde_temporal:.0%}, "
    f"semilla: {config_split.semilla}"
)

## Construcción del split

Se usa `torchvision.datasets.ImageFolder` para respetar el mismo orden de muestras y las mismas clases que consume el entrenamiento. `CLASES_A_PROCESAR` permite limitar el split a un subconjunto de carpetas/clases. Luego se aplica `train_test_split` dos veces: primero `train` contra el conjunto temporal, y después `val` contra `test`.


In [ ]:
dataset_completo = datasets.ImageFolder(DATA_ORIGEN)

if CLASES_A_PROCESAR is None:
    clases_a_procesar = set(dataset_completo.classes)
else:
    clases_a_procesar = set(CLASES_A_PROCESAR)
    clases_inexistentes = sorted(clases_a_procesar - set(dataset_completo.classes))
    if clases_inexistentes:
        raise ValueError(
            "Las siguientes clases no existen en DATA_ORIGEN: "
            f"{clases_inexistentes}. Clases disponibles: {dataset_completo.classes}"
        )

indices_clases = {dataset_completo.class_to_idx[clase] for clase in clases_a_procesar}
muestras_filtradas = [
    (ruta, etiqueta)
    for ruta, etiqueta in dataset_completo.samples
    if etiqueta in indices_clases
]

if not muestras_filtradas:
    raise ValueError("No hay imagenes para las clases seleccionadas")

dataset_completo.samples = muestras_filtradas
dataset_completo.imgs = muestras_filtradas
dataset_completo.targets = [etiqueta for _, etiqueta in muestras_filtradas]

indices = np.arange(len(dataset_completo))
etiquetas = np.array(dataset_completo.targets)

train_idx, temp_idx, _, etiquetas_temporales = train_test_split(
    indices,
    etiquetas,
    test_size=config_split.proporcion_temporal,
    stratify=etiquetas,
    random_state=config_split.semilla,
)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=config_split.proporcion_test_desde_temporal,
    stratify=etiquetas_temporales,
    random_state=config_split.semilla,
)

split_por_indice = {
    **{int(indice): "train" for indice in train_idx},
    **{int(indice): "val" for indice in val_idx},
    **{int(indice): "test" for indice in test_idx},
}

assert len(split_por_indice) == len(
    dataset_completo
), "Hay indices duplicados o faltantes en el split"

clases_procesadas = [
    clase for clase in dataset_completo.classes if clase in clases_a_procesar
]
print("Clases procesadas:", clases_procesadas)
print("Imagenes procesadas:", len(dataset_completo))
print(
    f"Split -> train: {len(train_idx)}, "
    f"val: {len(val_idx)}, "
    f"test: {len(test_idx)}"
)

In [5]:
# Tabla de archivos con split
registros_split = []
for indice, (ruta_imagen, etiqueta) in enumerate(dataset_completo.samples):
    ruta_imagen = Path(ruta_imagen)
    ruta_imagen = (
        ruta_imagen if ruta_imagen.is_absolute() else DATA_ORIGEN / ruta_imagen
    )
    ruta_relativa = ruta_imagen.relative_to(DATA_ORIGEN)
    registros_split.append(
        {
            "Identificador": ruta_imagen.stem,
            "archivo": ruta_imagen.name,
            "ruta_relativa": ruta_relativa.as_posix(),
            "clase_carpeta": dataset_completo.classes[etiqueta],
            "split": split_por_indice[indice],
        }
    )

df_split = pd.DataFrame(registros_split)
df_split.head()

,Identificador,archivo,ruta_relativa,clase_carpeta,split
0,al_13b,al_13b.png,clase01/al_13b.png,clase01,train
1,al_14b,al_14b.png,clase01/al_14b.png,clase01,train
2,ba_115_4,ba_115_4.png,clase01/ba_115_4.png,clase01,train
3,ba_172_3,ba_172_3.png,clase01/ba_172_3.png,clase01,train
4,ba_191_2,ba_191_2.png,clase01/ba_191_2.png,clase01,train


## Union con metadata

La metadata se limita a las imagenes procesadas y se agregan las columnas `archivo`, `ruta_relativa`, `clase_carpeta` y `split`.


In [ ]:
df_metadata = METADATA_ORIGEN.copy()

if "split" in df_metadata.columns:
    df_metadata = df_metadata.drop(columns=["split"])

ids_imagenes = set(df_split["Identificador"])
ids_metadata_origen = set(df_metadata["Identificador"])
ids_sin_metadata = sorted(ids_imagenes - ids_metadata_origen)
if ids_sin_metadata:
    print(
        "Advertencia: hay imagenes sin registro en metadata. "
        f"Cantidad: {len(ids_sin_metadata)}. Primeros casos: {ids_sin_metadata[:10]}"
    )

# Si se seleccionaron clases, se conserva solo la metadata de las imagenes procesadas.
df_metadata = df_metadata[df_metadata["Identificador"].isin(ids_imagenes)].copy()

columnas_derivadas = ["archivo", "ruta_relativa", "clase_carpeta", "split"]
df_metadata_split = df_metadata.merge(
    df_split[["Identificador", *columnas_derivadas]],
    on="Identificador",
    how="left",
    validate="one_to_one",
)

faltantes_en_imagenes = df_metadata_split["split"].isna().sum()
if faltantes_en_imagenes:
    ids_faltantes = df_metadata_split.loc[
        df_metadata_split["split"].isna(), "Identificador"
    ].tolist()
    raise ValueError(
        "Hay registros de metadata sin imagen procesada en DATA_CUADRADOS: "
        f"{ids_faltantes[:10]}"
    )

assert df_metadata_split["split"].notna().all()
df_metadata_split.head()

In [ ]:
# Control de distribucion por split y clase
resumen_split = pd.crosstab(
    df_metadata_split["clase_carpeta"],
    df_metadata_split["split"],
    margins=True,
)
resumen_split

In [ ]:
# Guardado de metadata reutilizable por otros notebooks
ruta_metadata_split = guardar_metadata(
    df=df_metadata_split,
    carpeta_destino=DATA_DESTINO,
)

print(f"Metadata con split guardada en: {ruta_metadata_split}")
print(f"Registros guardados: {len(df_metadata_split)}")

## Uso desde otros notebooks

```python
import pandas as pd

from vasijas.config import load_config
from vasijas.paths import ProjectPaths

CONFIG = load_config()
PATHS = ProjectPaths.from_root(CONFIG.project_root)

metadata_split = pd.read_csv(PATHS.DATA_DIVIDIDO / "metadata.csv")
train_df = metadata_split[metadata_split["split"] == "train"]
val_df = metadata_split[metadata_split["split"] == "val"]
test_df = metadata_split[metadata_split["split"] == "test"]
```
